In [0]:
from pyspark.sql.functions import col, expr, trim, upper

# -----------------------------
# 1. Read data
# -----------------------------
df = spark.read.table("mini_project_catalog.`01_bronze`.store")

# -----------------------------
# 2. Remove duplicates
# -----------------------------
df = df.dropDuplicates()

# -----------------------------
# 3. Replace invalid values ("-") with null
# -----------------------------
df = df.replace("-", None)

# -----------------------------
# 4. Trim all string columns
# -----------------------------
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# -----------------------------
# 5. Convert numeric columns
# -----------------------------
if "store_id" in df.columns:
    df = df.withColumn("store_id", col("store_id").cast("int"))

if "opened_year" in df.columns:
    df = df.withColumn("opened_year", col("opened_year").cast("int"))

# -----------------------------
# 6. Standardize text columns (IMPORTANT)
# -----------------------------
df = df.withColumn("store_type", upper(col("store_type")))
df = df.withColumn("state", upper(col("state")))

# -----------------------------
# 7. Filter valid records
# -----------------------------
df = df.filter(col("store_id").isNotNull())
df = df.filter(col("store_name").isNotNull())

# -----------------------------
# 8. Write to Silver (KEEP ALL COLUMNS)
# -----------------------------
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("mini_project_catalog.02_silver.store")

In [0]:
%sql
SELECT * FROM mini_project_catalog.02_silver.store LIMIT 10;